In [26]:
!pip install -q tf-keras-vis opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.5/52.5 kB 1.7 MB/s eta 0:00:00


In [27]:
import os
import cv2
import random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from google.colab import drive

from tensorflow.keras.models import load_model

from tf_keras_vis.gradcam import Gradcam
from tf_keras_vis.utils.scores import CategoricalScore
from tf_keras_vis.utils.model_modifiers import ReplaceToLinear

In [28]:
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [29]:
PROJECT_DIR="/content/drive/MyDrive/computervisionproject"

MODEL_PATH=os.path.join(
    PROJECT_DIR,
    "models",
    "best_model.keras"
)

TEST_DIR=os.path.join(
    PROJECT_DIR,
    "dataset",
    "processed_dataset",
    "test"
)

SAVE_DIR=os.path.join(
    PROJECT_DIR,
    "reports",
    "gradcam"
)

os.makedirs(SAVE_DIR,exist_ok=True)

In [30]:
model=load_model(MODEL_PATH)

print("Model Loaded")

model.summary()

Model Loaded


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 3)              │         3,843 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,053,422 (26.91 MB)

 Trainable params: 1,500,003 (5.72 MB)

 Non-trainable params: 2,553,411 (9.74 MB)

 Optimizer params: 3,000,008 (11.44 MB)

In [31]:
CLASS="COVID"

folder=os.path.join(TEST_DIR,CLASS)

image_name=random.choice(os.listdir(folder))

IMAGE_PATH=os.path.join(folder,image_name)

print(IMAGE_PATH)

/content/drive/MyDrive/computervisionproject/dataset/processed_dataset/test/COVID/COVID-1113.png


In [32]:
IMG_SIZE=224

img=cv2.imread(IMAGE_PATH)

img=cv2.cvtColor(
    img,
    cv2.COLOR_BGR2RGB
)

original=img.copy()

img=cv2.resize(
    img,
    (IMG_SIZE,IMG_SIZE)
)

input_image=np.expand_dims(
    img.astype(np.float32),
    axis=0
)

In [33]:
CLASS_NAMES=[
    "COVID",
    "Normal",
    "Viral Pneumonia"
]

prediction=model.predict(
    input_image,
    verbose=0
)

predicted_index=np.argmax(prediction)

predicted_class=CLASS_NAMES[predicted_index]

confidence=prediction[0][predicted_index]

print(predicted_class)

print(confidence)

COVID
0.9879556


In [34]:
replace2linear=ReplaceToLinear()

In [35]:
gradcam=Gradcam(

    model,

    model_modifier=replace2linear,

    clone=True

)

In [36]:
score = CategoricalScore(predicted_index)

In [37]:
cam = gradcam(
    score,
    input_image,
    penultimate_layer="top_conv"
)

heatmap = cam[0]

KeyError: "Exception encountered when calling Functional.call().\n\n\x1b[1m137113153337680\x1b[0m\n\nArguments received by Functional.call():\n  • inputs=['tf.Tensor(shape=(1, 224, 224, 3), dtype=float32)']\n  • training=False\n  • mask=['None']\n  • kwargs=<class 'inspect._empty'>"

In [ ]:
heatmap = np.maximum(heatmap, 0)

heatmap = heatmap / (heatmap.max() + 1e-8)

plt.figure(figsize=(6,6))

plt.imshow(
    heatmap,
    cmap="jet"
)

plt.title("Grad-CAM Heatmap")

plt.axis("off")

plt.show()

In [ ]:
heatmap = cv2.resize(
    heatmap,
    (
        original.shape[1],
        original.shape[0]
    )
)

heatmap = np.uint8(255 * heatmap)

heatmap = cv2.applyColorMap(
    heatmap,
    cv2.COLORMAP_JET
)

heatmap = cv2.cvtColor(
    heatmap,
    cv2.COLOR_BGR2RGB
)

In [ ]:
overlay = cv2.addWeighted(
    original,
    0.65,
    heatmap,
    0.35,
    0
)

In [ ]:
plt.figure(figsize=(18,6))

plt.subplot(1,3,1)

plt.imshow(original)

plt.title("Original")

plt.axis("off")


plt.subplot(1,3,2)

plt.imshow(heatmap)

plt.title("Grad-CAM")

plt.axis("off")


plt.subplot(1,3,3)

plt.imshow(overlay)

plt.title(
    f"{predicted_class}\nConfidence: {confidence:.2%}"
)

plt.axis("off")

plt.tight_layout()

plt.show()

In [ ]:
save_path = os.path.join(
    SAVE_DIR,
    f"{predicted_class}_gradcam.png"
)

cv2.imwrite(
    save_path,
    cv2.cvtColor(
        overlay,
        cv2.COLOR_RGB2BGR
    )
)

print("Saved to")

print(save_path)

In [ ]:
for cls in CLASS_NAMES:

    folder = os.path.join(TEST_DIR, cls)

    img_name = random.choice(os.listdir(folder))

    print(cls, "->", img_name)

In [ ]:
plt.figure(figsize=(8,4))

plt.bar(
    CLASS_NAMES,
    prediction[0]
)

plt.ylabel("Probability")

plt.title("Prediction Confidence")

plt.show()

In [ ]:
print("="*60)

print("Grad-CAM COMPLETED")

print("="*60)

print("Prediction :", predicted_class)

print(f"Confidence : {confidence:.2%}")

print()

print("Saved Image :")

print(save_path)

print("="*60)

In [38]:
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("Keras:", tf.keras.__version__)

TensorFlow: 2.20.0
Keras: 3.13.2


In [39]:
print(type(model))
print(model.inputs)
print(model.outputs)

print()

for layer in model.layers:
    print(layer.name, type(layer))

<class 'keras.src.models.functional.Functional'>
[<KerasTensor shape=(None, 224, 224, 3), dtype=float32, sparse=False, ragged=False, name=input_layer_1>]
[<KerasTensor shape=(None, 3), dtype=float32, sparse=False, ragged=False, name=keras_tensor_1561>]

input_layer_1 <class 'keras.src.layers.core.input_layer.InputLayer'>
sequential <class 'keras.src.models.sequential.Sequential'>
efficientnetb0 <class 'keras.src.models.functional.Functional'>
global_average_pooling2d <class 'keras.src.layers.pooling.global_average_pooling2d.GlobalAveragePooling2D'>
dropout <class 'keras.src.layers.regularization.dropout.Dropout'>
dense <class 'keras.src.layers.core.dense.Dense'>
